# Identity Anomaly Detection System: Technical Walkthrough

**Objective:** Demonstrate the end-to-end pipeline of the Identity Anomaly Detection System, from raw logs to ensemble risk scoring.

---

## 1. Setup & Data Loading
First, we load the raw authentication logs. This dataset represents 30 days of enterprise traffic.

In [ ]:
import pandas as pd
import numpy as np
from src.data_pipeline import DataPipeline
from src.ensemble import EnsembleDetector

# Initialize Pipeline
pipeline = DataPipeline()

# Load Data
raw_df = pipeline.ingest_csv('auth_logs.csv')
print(f"Dataset Shape: {raw_df.shape}")
raw_df.head()

## 2. Feature Engineering
Raw logs (timestamps, IPs) are converted into mathematical features for the models.

**Key Features:**
*   `hour` & `is_night`: Time-based context.
*   `failed_attempts`: Behavioral velocity.
*   `download_mb`: Data egress volume.

In [ ]:
processed_df = pipeline.engineer_features(raw_df)
X = pipeline.get_feature_matrix(processed_df)

# Show the "Math" the model sees
print("Feature Matrix (First 5 rows):")
print(X[:5])

## 3. Model Training (The Ensemble)
We train both the **Isolation Forest** (for outliers) and **Autoencoder** (for pattern matching) simultaneously.

In [ ]:
detector = EnsembleDetector()
print("Training Ensemble Model... (Isolation Forest + Autoencoder)")
detector.train(X, y=None) # Unsupervised training
print("Training Complete.")

## 4. The Ensemble Logic (Prediction)
Here is the core logic where we combine the models using a **Weighted Average**.

In [ ]:
def calculate_risk_score(model, features):
    # 1. Get Scores from both models
    _, score_if = model.isolation_forest.predict_with_scores(features)
    _, score_ae = model.autoencoder.predict_with_scores(features)
    
    # 2. Normalize
    s_if = score_if / 100.0
    s_ae = score_ae / 100.0
    
    # 3. Apply Weighted Formula
    # Risk = (Weight_IF * S_IF) + (Weight_AE * S_AE)
    ensemble_score = (
        (model.weights['isolation_forest'] * s_if) +
        (model.weights['autoencoder'] * s_ae)
    )
    
    return ensemble_score * 100

# Test with a sample event
sample_event = X[0].reshape(1, -1)
risk = calculate_risk_score(detector, sample_event)
print(f"Calculated Risk Score: {risk[0]:.2f}/100")

## 5. Adversarial Simulation
Let's simulate a **Brute Force Attack** (High failed attempts) and see if the model catches it.

In [ ]:
# Create a Fake Attack Vector
# [hour, is_weekend, ..., failed_attempts (high), ...]
attack_vector = np.array([[3, 1, 1, 50, 500, 0, 1, 0, 1, 0]]) 

prediction = detector.predict(attack_vector)
risk_score = detector.predict_with_scores(attack_vector)[1]

print(f"Attack Scenario: Brute Force Attempt")
print(f"Ensemble Detection: {'ANOMALY' if prediction[0] == 1 else 'NORMAL'}")
print(f"Risk Score: {risk_score[0]}/100")